# LightGBM: Hyperparameter Tuning with Optuna

This notebook focus on optimizing the LightGBM classifier using Bayesian search for multiclass churn prediction.

In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import joblib
import os
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [17]:
def load_data():
    logger.info("Loading split data...")
    X_train = pd.read_csv('../../data/04_features/X_train.csv')
    y_train = pd.read_csv('../../data/04_features/y_train.csv').squeeze().astype(int)
    X_test = pd.read_csv('../../data/04_features/X_test.csv')
    y_test = pd.read_csv('../../data/04_features/y_test.csv').squeeze().astype(int)
    return X_train, X_test, y_train, y_test

X_train_full, X_test, y_train_full, y_test = load_data()

# Create a validation set for Optuna and Early Stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

logger.info(f"Train set: {X_train.shape}, Val set: {X_val.shape}, Test set: {X_test.shape}")

2026-04-15 11:14:01,599 - INFO - Loading split data...
2026-04-15 11:14:01,723 - INFO - Train set: (4096, 16), Val set: (1025, 16), Test set: (1281, 16)


In [18]:
def objective(trial):
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }

    model = lgb.LGBMClassifier(**param)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False)
        ]
    )

    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average='macro')
    return score

In [19]:
logger.info("Starting optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

logger.info("Optimization finished.")
logger.info(f"Best trial score (Macro F1): {study.best_value}")
logger.info(f"Best parameters: {study.best_params}")

2026-04-15 11:14:01,754 - INFO - Starting optimization...
[I 2026-04-15 11:14:01,757] A new study created in memory with name: no-name-bff352ce-ea97-4e26-848d-d8878ed17657
[I 2026-04-15 11:14:02,030] Trial 0 finished with value: 0.7558536110231217 and parameters: {'n_estimators': 598, 'learning_rate': 0.0974660359680649, 'max_depth': 3, 'num_leaves': 16, 'min_child_samples': 49, 'subsample': 0.7647999963749004, 'colsample_bytree': 0.7349131511638746, 'reg_alpha': 0.9338934126679427, 'reg_lambda': 2.720406895518399}. Best is trial 0 with value: 0.7558536110231217.
[I 2026-04-15 11:14:02,327] Trial 1 finished with value: 0.7586357042490915 and parameters: {'n_estimators': 757, 'learning_rate': 0.055155279028408714, 'max_depth': 3, 'num_leaves': 10, 'min_child_samples': 100, 'subsample': 0.6458412727121132, 'colsample_bytree': 0.8076913186566573, 'reg_alpha': 3.6647280653961993, 'reg_lambda': 0.8434034960821102}. Best is trial 1 with value: 0.7586357042490915.
[I 2026-04-15 11:14:03,122] 

In [20]:
def train_final_model(best_params, X_train, y_train, X_validation, y_validation):
    logger.info("Training final model with best parameters...")
    
    final_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'random_state': 42,
        'class_weight': 'balanced',
        **best_params
    }
    
    model = lgb.LGBMClassifier(**final_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric='multi_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    return model

# Retrain on full training data, using test data for early stopping validation
final_model = train_final_model(study.best_params, X_train_full, y_train_full, X_test, y_test)

2026-04-15 11:14:20,586 - INFO - Training final model with best parameters...


Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.632741
[100]	valid_0's multi_logloss: 0.477986
[150]	valid_0's multi_logloss: 0.417363
[200]	valid_0's multi_logloss: 0.392183
[250]	valid_0's multi_logloss: 0.381569
[300]	valid_0's multi_logloss: 0.376793
[350]	valid_0's multi_logloss: 0.374118
[400]	valid_0's multi_logloss: 0.372534
[450]	valid_0's multi_logloss: 0.371307
[500]	valid_0's multi_logloss: 0.370468
[550]	valid_0's multi_logloss: 0.369669
[600]	valid_0's multi_logloss: 0.369022
[650]	valid_0's multi_logloss: 0.368756
[700]	valid_0's multi_logloss: 0.368298
[750]	valid_0's multi_logloss: 0.368033
Did not meet early stopping. Best iteration is:
[766]	valid_0's multi_logloss: 0.367966


In [21]:
def evaluate_model(model, X_test, y_test):
    logger.info("Evaluating final model...")
    preds = model.predict(X_test)
    
    print(f"\nAccuracy: {accuracy_score(y_test, preds)}")
    print("\nClassification Report:\n", classification_report(y_test, preds))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
    
    logger.info("Saving model...")
    os.makedirs('../../models', exist_ok=True)
    joblib.dump(model, '../../models/lightgbm_churn_model.pkl')
    logger.info("Model saved to ../../models/lightgbm_churn_model.pkl")

evaluate_model(final_model, X_test, y_test)

2026-04-15 11:14:21,911 - INFO - Evaluating final model...
2026-04-15 11:14:22,018 - INFO - Saving model...
2026-04-15 11:14:22,130 - INFO - Model saved to ../../models/lightgbm_churn_model.pkl



Accuracy: 0.810304449648712

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.62      0.69       427
           1       0.47      0.69      0.55       214
           2       1.00      0.98      0.99       640

    accuracy                           0.81      1281
   macro avg       0.75      0.76      0.75      1281
weighted avg       0.84      0.81      0.82      1281


Confusion Matrix:
 [[266 161   0]
 [ 67 147   0]
 [  7   8 625]]
